In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import LeaveOneOut, LeaveOneGroupOut
from sklearn.metrics import r2_score
import io, warnings
warnings.filterwarnings("ignore", category=UserWarning)

# Mirror of app.py. Same CSV, same 9 features, same parameters, so the printed
# numbers match the README and the deployed Streamlit app.
#
# spotify_mau = Spotify MAU (millions), annual figure, same for all albums in a year
# 2015=80  2016=100  2017=120  2018=207  2019=248  2020=299
# 2021=365 2022=489  2023=574  2024=640  2025=715
# 2026 Jan-Jun = 761,  2026 Jul-Dec = 788  (Q3 2026 earnings)
#
# hype_score was removed from the data. It was scored after those albums released, so it
# carried hindsight the model could exploit. See "Model Revision, October 2026" in the README.
# lead_time = days from the first official announcement of the release date to the release.
# Every value was re-checked against announcement dates in October 2026.

csv_content = """artist,album,year,track_count,spotify_mau,lead_time,feature_track_count,years_since_last_album,prev_album_streams,first_week_streams,is_household_name
Harry Styles,HarryStyles,2017,10,120,29,0,0,0,66,0
Harry Styles,FineLine,2019,12,248,39,0,2,66,155,1
Harry Styles,HarrysHouse,2022,13,489,58,0,2,155,440,1
Harry Styles,KissAllTheTime,2026,14,761,50,0,4,440,205,1
Noah Kahan,TheGreatDivide,2026,17,761,86,0,4,0,231.5,0
Kanye West,BULLY,2026,18,761,3,6,2,251,158.9,1
Don Toliver,OCTANE,2026,18,761,8,5,2,0,162.5,0
J Cole,TheFallOff,2026,24,761,23,7,5,207.1,139.1,1
Bruno Mars,TheRomantic,2026,9,761,51,0,5,0,144.0,1
A$AP Rocky,DontBeDumb,2026,15,761,27,6,8,0,129.7,0
Zach Bryan,WithHeavenOnTop,2026,25,761,175,0,2,98.5,98.1,0
Ella Langley,Dandelion,2026,18,761,73,1,2,0,88.4,0
Karol G,NoMeArrepiento,2026,14,788,9,4,1,136.2,76.4,1
Olivia Rodrigo,YSPSFAGSIL,2026,13,761,71,1,3,284,394.7,1
Drake,Iceman,2026,18,761,24,2,3,405,450.4,1
BTS,Arirang,2026,14,761,74,0,6,153.8,525.6,1
Ariana Grande,Petal,2026,12,788,94,0,2,342,236.1,1
Ariana Grande,DangerousWoman,2016,15,100,71,4,2,0,53.6,1
Ariana Grande,Sweetener,2018,15,207,58,2,2,53.6,174,1
Ariana Grande,ThanksUNext,2019,12,248,17,0,1,174,361,1
Ariana Grande,Positions,2020,14,299,7,3,1,361,216.7,1
Ariana Grande,EternalSunshine,2024,13,640,51,0,4,216.7,342,1
Taylor Swift,Lover,2019,18,248,71,2,2,0,257.1,1
Taylor Swift,Folklore,2020,16,299,1,1,1,257.1,329,1
Taylor Swift,Evermore,2020,15,299,1,3,0,329,254,1
Taylor Swift,Midnights,2022,20,489,54,1,2,254,776,1
Taylor Swift,1989TV,2023,21,574,79,0,1,776,582,1
Taylor Swift,TTPD,2024,31,640,75,2,1,582,1173,1
Taylor Swift,LifeOfAShowgirl,2025,12,715,52,1,1,1173,887,1
Bad Bunny,UltimoTourDelMundo,2020,16,299,2,0,0,82.9,274,1
Bad Bunny,YHLQMDLG,2020,20,299,2,11,2,67.4,295.6,1
Bad Bunny,LasQueNoIbanASalir,2020,10,299,0,6,0,295.6,82.9,1
Bad Bunny,UnVeranoSinTi,2022,23,489,4,8,1,274,503,1
Bad Bunny,NadieSabe,2023,22,574,4,8,1,503,471,1
Bad Bunny,DeBITiRARMasFotos,2025,17,715,10,3,2,471,579.2,1
Tyler The Creator,Igor,2019,12,248,11,5,2,0,114.3,0
Tyler The Creator,CallMeIfYouGetLost,2021,16,365,8,12,2,114.3,133,0
Tyler The Creator,CHROMAKOPIA,2024,14,640,11,7,3,133,340,0
Tyler The Creator,DontTapTheGlass,2025,10,715,3,3,1,340,111.5,0
Billie Eilish,WWAFAWDWG,2019,14,248,59,0,0,0,276.3,0
Billie Eilish,HappierThanEver,2021,16,365,95,0,2,276.3,178,1
Billie Eilish,HMHAS,2024,10,640,39,0,3,178,386,1
Post Malone,Stoney,2016,18,100,36,4,0,0,14.5,0
Post Malone,BBandB,2018,18,207,23,5,2,14.5,411,1
Post Malone,HollywoodsBleeding,2019,17,248,11,7,1,411,379,1
Post Malone,TwelveCaratToothache,2022,14,489,37,6,3,379,142,1
Post Malone,Austin,2023,17,574,32,0,1,142,119.4,1
Post Malone,F1Trillion,2024,18,640,59,15,1,119.4,180.2,1
Ed Sheeran,Divide,2017,16,120,50,0,3,23.8,375,1
Ed Sheeran,Equals,2021,14,365,71,0,4,375,161,1
Drake,Scorpion,2018,25,207,16,3,1,290.2,559,1
Drake,CLB,2021,21,365,7,12,3,559,496,1
Drake,FATD,2023,23,574,14,10,2,496,405,1
Travis Scott,ASTROWORLD,2018,17,207,4,11,3,0,257,0
Travis Scott,Utopia,2023,19,574,6,13,5,257,457,0
Eminem,Revival,2017,19,120,17,8,4,10,117,1
Eminem,Kamikaze,2018,13,207,0,4,1,117,238,1
Eminem,TheDeathOfSlimShady,2024,19,640,11,8,6,238,254,1
The Weeknd,BeautyBehindTheMadness,2015,14,80,50,3,2,0,82.8,0
The Weeknd,Starboy,2016,18,100,65,5,1,82.8,223,1
The Weeknd,AfterHours,2020,14,299,30,0,2,223,265,1
The Weeknd,HUT,2025,22,715,65,7,4,265,302,1
Justin Bieber,Purpose,2015,13,80,42,4,3,0,205,1
Justin Bieber,Changes,2020,17,299,18,6,5,205,181,1
Justin Bieber,Justice,2021,16,365,21,8,1,181,252,1
Justin Bieber,Swag,2025,21,715,1,7,4,252,276.8,1
SZA,SOS,2022,23,489,6,4,5,125,336,0
Sabrina Carpenter,EmailsICantSend,2022,13,489,10,0,3,0,22.3,0
Sabrina Carpenter,ShortNSweet,2024,12,640,81,0,2,22.3,352,0
Sabrina Carpenter,MansBestFriend,2025,13,715,79,0,1,352,297,0
Kendrick Lamar,DAMN,2017,14,120,3,3,2,0,227,1
Kendrick Lamar,MrMorale,2022,19,489,25,7,5,227,343,1
Kendrick Lamar,GNX,2024,12,640,0,4,2,343,363,1
LilWayne,CarterV,2018,23,207,2,10,7,0,226.9,0
Olivia Rodrigo,SOUR,2021,11,365,50,0,0,0,385,1
Olivia Rodrigo,GUTS,2023,12,574,74,0,2,385,284,1
Beyonce,Renaissance,2022,16,489,43,2,6,0,173,1
Beyonce,CowboyCarter,2024,27,640,47,11,2,173,316,1
Dua Lipa,FutureNostalgia,2020,13,299,58,2,4,0,148.9,0
Dua Lipa,RadicalOptimism,2024,11,640,51,0,4,148.9,94,0
Adele,30,2021,12,365,37,0,6,0,310,1
Karol G,KG0516,2021,16,365,9,10,2,0,54.9,0
Karol G,MananaSeraBonito,2023,17,574,14,10,2,54.9,203,0
Karol G,Tropicoqueta,2025,20,715,10,6,2,203,136.2,0
21 Savage,IAmGreaterThanIWas,2018,15,207,10,8,1,0,88.6,0
21 Savage,AmericanDream,2024,15,640,3,10,2,381.7,187,0
Kanye West,JesusIsKing,2019,11,248,57,5,1,141.0,174.3,1
Kanye West,DONDA,2021,27,365,23,23,2,174.3,442,1
Kanye West,VULTURES1,2024,15,640,24,0,3,442,251,1
Tate McRae,ThinkLater,2023,14,574,32,0,2,0,115,0
Tate McRae,SoCloseToWhat,2025,16,715,99,2,2,115,188,0
Morgan Wallen,Dangerous,2021,30,365,58,2,3,0,161,0
Morgan Wallen,OneThingAtATime,2023,36,574,32,3,2,161,251.1,0
Morgan Wallen,ImTheProblem,2025,37,715,57,5,2,251.1,253,0
Juice WRLD,DeathRaceForLove,2019,22,248,29,3,1,0,126.3,0
Juice WRLD,LegendsNeverDie,2020,22,299,4,5,1,126.3,303,0
Playboi Carti,WholeLottaRed,2020,24,299,4,3,2,0,88.5,0
Playboi Carti,MUSIC,2025,30,715,2,12,5,88.5,483.35,0
Lady Gaga,Mayhem,2025,14,715,39,2,5,173,216.08,1
Future,INeverLikedYou,2022,16,489,4,6,2,0,118.6,0
Zach Bryan,ZachBryan,2023,16,574,16,4,1,0,145.5,0
Zach Bryan,TheGreatAmericanBarScene,2024,19,640,14,5,1,145.5,98.5,0
Khalid,FreeSpirit,2019,17,248,36,4,1,0,142.5,0
BTS,LoveYourselfTear,2018,11,207,32,1,0,0,72.3,1
BTS,MapOfTheSoul7,2020,20,299,45,2,1,98.1,144.4,1
BTS,BE,2020,8,299,53,0,0,144.4,153.8,1
Olivia Dean,TheArtOfLoving,2025,12,715,116,0,2,0,97,0
J Cole,KOD,2018,12,207,4,0,2,0,190,0
J Cole,TheOffSeason,2021,12,365,10,5,3,190,207.1,0
Rosalia,LUX,2025,18,715,18,5,3,79.0,182.9,0
J Balvin,Jose,2021,24,365,28,14,1,102.0,85.5,1
Gunna,DS4EVER,2022,19,489,6,11,2,0,85.8,0
Daddy Yankee,Legendaddy,2022,19,489,4,7,10,0,78.6,1
Anuel AA,Emmanuel,2020,22,299,2,12,2,0,78.3,0
Shakira,LasMujeresYaNoLloran,2024,15,640,36,10,7,0,67.5,1
Lil Uzi Vert,LuvIsRage2,2017,16,120,1,3,0,0,82.9,0
Lil Baby,MyTurn,2020,20,299,29,8,2,0,81.4,0
Twenty One Pilots,Trench,2018,14,207,86,0,3,0,76.4,0
Central Cee,CantRushGreatness,2025,17,715,85,6,3,0,96.2,0
sombr,IBarelyKnowHer,2025,10,715,11,0,0,0,98.7,0
Addison Rae,Addison,2025,12,715,44,0,2,0,63.8,0
Alex Warren,YoullBeAlrightKid,2025,21,715,45,2,1,0,133.8,0
Jennie,Ruby,2025,15,715,45,5,0,0,148.6,0
Sleep Token,EvenInArcadia,2025,10,715,57,0,2,0,82,0
Doja Cat,Vie,2025,15,715,43,1,2,124.5,56,0"""

# --- Reference-only history (NOT in training) ---------------------------------
# These two projects have real first-week numbers but were deliberately left out of the
# dataset (collaboration / project-type releases). Kept for documentation only.
#
# 21 Savage,SavageMode2,2020,15,299,4,5,2,0,108,0   (collab w/ Metro Boomin)
# Metro Boomin,HeroesVillains,2022,15,489,77,10,4,0,226.2,0   (solo album, verified; Metro has no other rows)
# Metro Boomin,WDTY,2024,17,640,14,5,2,226.2,257,0   (joint album with Future, excluded by the joint-album rule)

# 9 features. hype_score dropped (hindsight leak); the three correlated history
# features collapsed into history_per_mau. has_hist = 1 if a real previous-album
# first-week figure exists, 0 if the fallback value stands in for one.
FEATS = ['track_count', 'spotify_mau', 'lead_time', 'feature_track_count',
         'years_since_last_album', 'history_per_mau', 'has_hist',
         'is_household_name', 'household_x_tracks']

MAU_2026 = 788   # Jul-Dec 2026. Use 761 for a Jan-Jun release.

# --- prep ---
df = pd.read_csv(io.StringIO(csv_content)).sort_values(['artist','year']).reset_index(drop=True)
df['household_x_tracks'] = df['is_household_name'] * df['track_count']
df['spt'] = df['first_week_streams'] / df['track_count']
# Fallback for artists with no earlier album: the median of the TRAINING albums only (2015-2024),
# so no holdout or 2026 album influences a training feature.
_trn = df[df['year']<=2024]
gm, gm_spt = _trn['first_week_streams'].median(), _trn['spt'].median()
df['has_hist'] = (df['prev_album_streams'] > 0).astype(int)
df['prev_album_streams'] = df['prev_album_streams'].replace(0, gm)

def prior_mean(df, col, fallback):
    """Mean over the artist's STRICTLY EARLIER albums. Prevents leakage."""
    out = []
    for _, r in df.iterrows():
        prior = df[(df['artist']==r['artist']) & (df['year']<r['year'])]
        out.append(prior[col].mean() if len(prior) > 0 else fallback)
    return out

df['t_avg'] = prior_mean(df, 'first_week_streams', gm)
df['t_spt'] = prior_mean(df, 'spt', gm_spt)
# Platform-size adjustment. Dividing by MAU in full over-corrects: across the training albums,
# first-week streams grew roughly with MAU^0.4, not MAU^1. ERA_EXP = 0.25 was picked by
# leave-one-out cross-validation from {0, 0.25, 0.5, 0.75, 1} (3-seed average; the 2025 holdout
# was not used to choose). The differences between exponents are within noise at this
# sample size (see README). The columns keep their *_per_mau names for continuity.
ERA_EXP = 0.25
df['era_scale'] = df['spotify_mau'] ** ERA_EXP
for col, src in [('trailing_avg_per_mau','t_avg'), ('trailing_spt_per_mau','t_spt'),
                 ('prev_streams_per_mau','prev_album_streams')]:
    df[col] = df[src] / df['era_scale']

# Single history signal: last album and career average, averaged.
df['history_per_mau'] = (df['prev_streams_per_mau'] + df['trailing_avg_per_mau']) / 2
df['log_spm'] = np.log(df['first_week_streams'] / df['era_scale'])

train, hold, val26 = (df[df['year']<=2024].copy(),
                      df[df['year']==2025].copy(),
                      df[df['year']==2026].copy())

# --- LOOCV ---
# Ridge regression on standardized features, alpha fixed at 1.0 in advance (sklearn's default,
# not tuned). Chosen over a Random Forest by cross-validation alone; see the README.
ALPHA = 1.0
def make_model():
    return make_pipeline(StandardScaler(), Ridge(alpha=ALPHA))

X, y = train[FEATS], train['log_spm']
lp, la = [], []
for ti, te in LeaveOneOut().split(X):
    m = make_model(); m.fit(X.iloc[ti], y.iloc[ti])
    lp.append(m.predict(X.iloc[te])[0]); la.append(y.iloc[te[0]])

r2     = r2_score(la, lp)
rp     = np.exp(np.array(lp)) * train['era_scale'].values
ra     = train['first_week_streams'].values
mape   = np.mean(np.abs(rp - ra) / ra * 100)
medape = np.median(np.abs(rp - ra) / ra * 100)
mae    = np.mean(np.abs(rp - ra))
r2_raw = r2_score(ra, rp)   # same predictions, scored on the raw stream scale

# Naive benchmark: always guess the median first-week total of the other albums.
base_pred = np.array([np.median(np.delete(ra, i)) for i in range(len(ra))])
base_ape  = np.abs(base_pred - ra) / ra * 100

# Leave-one-artist-out: every album by the held-out artist is removed from training.
lq = np.zeros(len(train))
for ti, te in LeaveOneGroupOut().split(X, y, train['artist'].values):
    m = make_model(); m.fit(X.iloc[ti], y.iloc[ti]); lq[te] = m.predict(X.iloc[te])
loao_med = np.median(np.abs(np.exp(lq) * train['era_scale'].values - ra) / ra * 100)

model = make_model()
model.fit(X, y)

# --- helpers ---
def predict(params, mau):
    """Predicted first-week streams (M). The model predicts log(streams / MAU**ERA_EXP), the same
    quantity the cross-validation numbers are computed on, then converts back."""
    Xi = pd.DataFrame([params], columns=FEATS)
    return float(np.exp(model.predict(Xi)[0]) * mau ** ERA_EXP)

def artist_params(artist, tracks, lead, feats, years, mau, before_year=2026):
    """Build a feature row for a new album. before_year restricts the artist's history
    to strictly earlier albums, matching how the training features were built."""
    full = df[df['artist']==artist].sort_values('year')
    a = full[full['year'] < before_year]
    is_hh = int(full.iloc[-1]['is_household_name']) if len(full) else 0
    prev = (a.iloc[-1]['first_week_streams'] if len(a) else gm) / mau ** ERA_EXP
    tavg = (a['first_week_streams'].mean()   if len(a) else gm) / mau ** ERA_EXP
    has_hist = int(len(a) > 0)
    return [tracks, mau, lead, feats, years, (prev + tavg) / 2, has_hist, is_hh, is_hh * tracks]

# --- 2026 case study: predicted ahead of each release -------------------------
# LOGGED values from the original model, kept as a record of what was actually
# predicted before each album came out. They are NOT recomputed here. The model has
# since been revised and would give different figures today; recomputing them would
# turn a forecast into a retrodiction.
CASE_STUDY = [
    ("DRAKE - ICEMAN",              479, 450.4, 22),
    ("OLIVIA RODRIGO - YSPSFAGSIL", 413, 394.7, 13),
    ("ARIANA GRANDE - PETAL",       408, 236.1, 12),
]

BOLD, BLUE, GREEN, RED, AMBER, RESET = "\033[1m", "\033[34m", "\033[32m", "\033[31m", "\033[33m", "\033[0m"
COL_W = [32, 9, 9, 21, 10, 9]

def fmt_row(label, m, tc, mape, actual):
    lo, hi = int(m*(1-mape/100)), int(m*(1+mape/100))
    d = (m - actual) / actual * 100
    c = GREEN if abs(d) < 15 else (AMBER if abs(d) < 30 else RED)
    return (label.ljust(COL_W[0])
            + (f"{BOLD}{BLUE}{int(m)}M{RESET}").ljust(COL_W[1] + len(BOLD+BLUE+RESET))
            + f"{m/tc:.1f}M".ljust(COL_W[2])
            + f"{lo}M-{hi}M".ljust(COL_W[3])
            + (f"{BOLD}{GREEN}{actual:.1f}M{RESET}").ljust(COL_W[4] + len(BOLD+GREEN+RESET))
            + f"{c}{BOLD}{d:+.1f}%{RESET}")

W = 100
hdr = ["ARTIST & ALBUM", "MODEL", "SPT", "RANGE (+/-" + f"{medape:.0f}" + "% typical)", "ACTUAL", "% DIFF"]

print("\n" + "="*W)
print(f"{'SUMMER 2026 CASE STUDY  (predicted ahead of each release)':^{W}}")
print("="*W)
print("  " + "".join(h.ljust(w) for h, w in zip(hdr, COL_W)))
print("  " + "-"*sum(COL_W))
for label, pred, actual, tc in CASE_STUDY:
    print("  " + fmt_row(label, pred, tc, medape, actual))
print("\n  Logged with the original 11-feature model. Not recomputed.")
print("  Two of three landed within 7%.")

print("\n" + "="*W + f"\n2025 HOLDOUT  ({len(hold)} albums, never seen in training)\n" + "="*W)
print(f"  {'ALBUM':<38}  {'PRED':>5}  {'ACTUAL':>6}  {'ERR':>9}\n  " + "-"*68)
herrs = []
for _, r in hold.iterrows():
    p = predict(r[FEATS].tolist(), r['spotify_mau'])
    e = abs(p - r['first_week_streams']) / r['first_week_streams'] * 100
    herrs.append(e)
    over = 'over' if p > r['first_week_streams'] else 'under'
    print(f"  {r['artist']+' '+r['album']:<38}  {int(p):>4}M  {int(r['first_week_streams']):>5}M  {e:>7.1f}% ({over})")
hbase = np.mean(np.abs(np.median(ra) - hold['first_week_streams'].values) / hold['first_week_streams'].values * 100)
hbase_med = np.median(np.abs(np.median(ra) - hold['first_week_streams'].values) / hold['first_week_streams'].values * 100)
print(f"  >> Holdout typical error: {np.median(herrs):.1f}%   (always guessing the median album: {hbase_med:.1f}%)")
print(f"  >> Holdout MAPE: {np.mean(herrs):.1f}%   (always guessing the median album: {hbase:.1f}%)")

print("\n" + "="*W + "\n2026 TEST SET  (frozen model, known results, never trained on)\n" + "="*W)
verrs, vbase = [], []
for _, r in val26.iterrows():
    p = predict(r[FEATS].tolist(), r['spotify_mau'])
    e = abs(p - r['first_week_streams']) / r['first_week_streams'] * 100
    verrs.append(e)
    vbase.append(abs(np.median(ra) - r['first_week_streams']) / r['first_week_streams'] * 100)
    over = 'over' if p > r['first_week_streams'] else 'under'
    print(f"  {r['artist']+' '+r['album']:<38}  {int(p):>4}M  {int(r['first_week_streams']):>5}M  {e:>7.1f}% ({over})")
print(f"  >> 2026 typical error: {np.median(verrs):.1f}%   (always guessing the median album: {np.median(vbase):.1f}%)")
print(f"  >> 2026 MAPE: {np.mean(verrs):.1f}%   (always guessing the median album: {np.mean(vbase):.1f}%)")
print(f"  Note: {len(val26)} albums. Inputs were locked before the model was run; nothing was tuned on them.")
print("  The 2025 holdout above was viewed during development; this set was not.")

print("\n" + "="*W + "\nFEATURE WEIGHTS  (Ridge coefficients on standardized features; + raises the prediction)\n" + "="*W)
for f, v in pd.Series(model.named_steps['ridge'].coef_, index=FEATS).sort_values(key=abs, ascending=False).items():
    print(f"  {f:<25}  {v:+.3f}  " + ("+" if v > 0 else "-")*max(1, int(abs(v)*60)))

print("="*W)
print(f"\n  train={len(train)} rows | features={len(FEATS)} | R2(log)={r2:.3f} | "
      f"MAPE={mape:.1f}% | median={medape:.1f}% | MAE={mae:.0f}M")
print(f"  R2 on the raw stream scale = {r2_raw:.3f}. The {r2:.2f} above is measured on")
print("  log(streams/MAU). On the raw scale, a score at or below zero means no better than")
print("  always guessing the average album's stream count.")
print(f"  Benchmark, always guess the median album: MAPE={base_ape.mean():.1f}% | median={np.median(base_ape):.1f}%")
print(f"  Model:                                    MAPE={mape:.1f}% | median={medape:.1f}%")
print(f"  Leave-one-artist-out, typical error: {loao_med:.1f}%")
print("="*W + "\n")
